# Day 22 — HTTP and REST in Depth: Requests, Status Codes, Headers, Idempotency & Pagination (15 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: read and write HTTP fluently. By tonight you can explain every line of a request and a response, choose the right method and status code without guessing, and design a REST API that a customer's engineering team would sign off.

- Deliverable: day22/api_design.md (a REST design for the Acme content API you build from tomorrow) and day22/curl_drills.sh (10 drills against a local mock API), pushed. DSA in day22/dsa.py.

- Where this sits: Week 4, Phase 1 (Foundation), day 1 of 7. FastAPI starts tomorrow, and FastAPI is a thin layer over exactly these ideas: a route is a method plus a path, a response is a status code plus headers plus a body. Every FDE integration (CMS webhooks, CRM syncs, the Claude API itself) is HTTP underneath.

Time split (5h): the Week 4 split from the Day 21 plan.

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Five concepts below, typing every example; check-your-understanding questions |
| Core task | 2h | api_design.md (75 min), curl_drills.sh (45 min) |
| Catch-up | 1h | Day 15: schema.sql and queries.sql (Day 4 first if it is still open) |
| DSA | 0.5h | parse_query and cursor_page in dsa.py |

Setup (5 min): today you call a small local API, day22/mock_api.py, that behaves like a headless CMS API. It uses only the standard library. Start it in one terminal and keep it running; use a second terminal for curl.

```bash
# One-time setup for Day 22, run inside day22/ (macOS / Linux shell).
source ../day06/.venv/bin/activate  # or any venv that has requests
python3 -m pip install requests      # skip if Day 6 already did this

# Terminal 1: start the mock API and leave it running.
python3 mock_api.py
# -> Acme mock API on http://127.0.0.1:8022  (Ctrl+C to stop)

# Terminal 2: check that it answers.
curl http://127.0.0.1:8022/health
# -> {
# ->   "status": "ok"
# -> }
```

Push rule for Week 4: push at the end of the day whatever state the work is in, for example "Day 22: WIP, api_design.md done, drills 1 to 6".

## Recap (2 minutes, from memory)

Three questions from Week 3, since yesterday was the review.

- You load 5,000 rows in one transaction with psycopg, and row 4,000 fails a CHECK constraint. What is in the table afterwards, and why is that the behaviour you want?

- What does INSERT ... ON CONFLICT (uid) DO UPDATE SET title = EXCLUDED.title do, and what is EXCLUDED?

- A table has 5 rows and 2 of them have views set to NULL. What do count(*) and count(views) return?

Answers are on the last page of today's session.

## Concept 1: What HTTP is, and how to see it with curl

### Why it matters for an FDE

When an integration breaks at a customer site, the first question is always "what exactly was sent, and what exactly came back?" The customer says "the sync fails"; you need the request line, the headers and the status code. Engineers who can read raw HTTP find problems in minutes. Engineers who only know a client library guess for hours.

You have used HTTP for 14 years through fetch(), Axios and the Contentstack SDKs. Today you look underneath those tools, because tomorrow you are on the other side: you write the server that answers.

### How it works

HTTP is a request and response protocol. A client opens a connection to a server, sends one request, and gets one response. The messages are plain text with a fixed layout (HTTP/2 and HTTP/3 send the same information in binary frames, but the meaning is identical, and every tool shows it to you in this text form).

- A request has a request line (method, path with query string, version), then headers (one Name: value per line), then a blank line, then an optional body.

- A response has a status line (version, status code, reason phrase), then headers, then a blank line, then an optional body.

- HTTP is stateless. The server does not remember your previous request. Everything it needs (who you are, what format you want, which version you last saw) travels in each request, usually in headers. That is why an API key goes on every call.

- HTTPS is the same HTTP inside a TLS-encrypted connection. Nothing in the message changes; only the wire is encrypted.

To prove it is only text, the next example skips every library and writes a request by hand over a raw network connection. You will never do this in production, but after doing it once, requests.get() stops being magic: it builds this text, sends it, and parses the reply.

In [ ]:
# HTTP is just text over a network connection. Send it by hand once.
import json
import socket  # low-level network connections; requests/fetch hide this

# 1. Open a TCP connection to the mock server (like dialling a number).
conn = socket.create_connection(("127.0.0.1", 8022))

# 2. A request is a request line, headers, then a blank line.
#    "\r\n" ends every line; the empty line means "headers are done".
request = (
    "GET /v1/entries/blt001 HTTP/1.1\r\n"  # method, path, HTTP version
    "Host: 127.0.0.1:8022\r\n"             # which site (required in 1.1)
    "X-API-Key: acme-dev-key\r\n"          # our credential (Concept 4)
    "Connection: close\r\n"                # close after one response
    "\r\n"
)
conn.sendall(request.encode())  # the network carries bytes, not str

# 3. Read everything the server sends back until it closes.
raw = b""
while chunk := conn.recv(4096):  # recv returns b"" when the server is done
    raw += chunk
conn.close()

head, body = raw.decode().split("\r\n\r\n", 1)  # blank line splits them
print(head)
# -> HTTP/1.1 200 OK                       <- status line
# -> Server: AcmeMock/1.0 Python/3.13.18
# -> Date: Sun, 15 Nov 2026 02:20:11 GMT
# -> X-Request-Id: req_5c1f0a9e            <- random on every call
# -> ETag: "v1"
# -> Cache-Control: max-age=60
# -> Content-Type: application/json
# -> Content-Length: 191                   <- body size in bytes
print(json.loads(body)["title"])  # -> Diwali sale guide

In daily work you use curl instead. It is on every laptop and server, it is what API docs (including Anthropic's and Contentstack's) show in their examples, and a curl command is the clearest way to hand a bug to someone else: they can run it and see what you saw.

```bash
# -v (verbose) shows the request curl sent (">") and the response
# headers it got back ("<"). This is the first thing to run when an
# integration "does not work".
curl -v http://127.0.0.1:8022/health
# Output, trimmed to the HTTP part:
# -> > GET /health HTTP/1.1            <- request line: method path version
# -> > Host: 127.0.0.1:8022            <- request headers start here
# -> > User-Agent: curl/8.5.0
# -> > Accept: */*
# -> >                                 <- blank line: request is complete
# -> < HTTP/1.1 200 OK                 <- status line: version code reason
# -> < Server: AcmeMock/1.0 Python/3.13.18
# -> < X-Request-Id: req_37a4f7bf      <- quote this id in bug reports
# -> < Content-Type: application/json  <- how to read the body
# -> < Content-Length: 20
# -> <                                 <- blank line, then the body
# -> {"status": "ok"}  (printed over three lines)
```

The curl options you will use all week:

| Option | What it does | JS / fetch equivalent |
|---|---|---|
| -i | Print response headers before the body | inspect res.headers |
| -v | Print the request and response headers (debugging) | the browser Network tab |
| -s | Silent: no progress bar | none |
| -X METHOD | Use this method (GET is the default) | method: "POST" |
| -H 'Name: value' | Add a request header (repeat for more) | headers: {...} |
| -d 'text' | Send a body (curl switches to POST) | body: JSON.stringify(...) |
| --json 'text' | Body plus JSON Content-Type and Accept headers (curl 7.82+) | body + headers together |
| -o FILE | Write the body to a file (/dev/null throws it away) | none |
| -w '%{http_code}' | Print values after the response, such as the status code | res.status |
| --fail | Exit with code 22 on 4xx and 5xx (useful in scripts, Day 9) | if (!res.ok) throw |

### Real-world FDE example

A retail customer reports that their nightly Contentstack-to-search sync "randomly fails". Their logs only say HTTPError. You ask for one failing call reproduced with curl -v. The output shows < HTTP/1.1 429 Too Many Requests and < Retry-After: 30: the sync is being rate limited at 2 a.m. when three jobs run at once. The fix is a retry that honours Retry-After and a staggered schedule, not a code rewrite. Ten minutes of reading headers saved a week of guessing.

### Common mistakes

- Debugging through a library's error message instead of looking at the raw status code and headers.

- Pasting a curl command with a real API key into Slack or a ticket. Replace it with $API_KEY before sharing.

- Forgetting that HTTP is stateless and expecting the server to "remember" a login or a previous page.

- Quoting: in the shell, wrap JSON bodies in single quotes so the double quotes inside reach curl unchanged.

## Concept 2: Resources, URLs and methods (the REST part)

### Why it matters for an FDE

REST is a set of conventions, not a standard, and that is exactly why it matters: when you design an API the way everyone expects, the customer's developers can use it from the docs alone. When you invent your own shape (POST /getEntries, GET /deleteEntry?id=3), every integration needs a meeting. Most customer-facing APIs you will meet (Contentstack's Management API, Salesforce, Stripe, GitHub) follow these conventions closely.

### How it works

A resource is a thing your API exposes, named with a noun: entries, authors, tags. Each resource has two kinds of URL: a collection (/v1/entries, all entries) and an item (/v1/entries/blt001, one entry, identified by its id). The method says what to do with it. In React terms, the URL is the state you are pointing at, and the method is the action you dispatch.

| Method | Meaning | Typical URL | Safe? | Idempotent? |
|---|---|---|---|---|
| GET | Read | /v1/entries, /v1/entries/blt001 | Yes | Yes |
| POST | Create in a collection, or run an action | /v1/entries, /v1/entries/blt001/publish | No | No |
| PUT | Replace the whole item with what I send | /v1/entries/blt001 | No | Yes |
| PATCH | Change only the fields I send | /v1/entries/blt001 | No | Not guaranteed |
| DELETE | Remove the item | /v1/entries/blt001 | No | Yes |

Two words in that table drive design decisions all week:

- Safe means the request does not change anything on the server. Caches, crawlers and retry logic assume GET is safe, so a GET that deletes or publishes something is a real bug, not a style issue.

- Idempotent means sending the same request twice leaves the server in the same state as sending it once. DELETE is idempotent: the second call may return 404 instead of 204, but the entry is gone either way. POST is not: two POSTs create two entries. Idempotency decides what a client may safely retry after a timeout (Concept 5).

Relationships become nested paths one level deep: /v1/entries/blt001/tags lists one entry's tags, and PUT /v1/entries/blt001/tags/sale attaches a tag (idempotent: attaching twice changes nothing). Actions that are not plain create, read, update or delete become a sub-resource you POST to, such as /publish; this is the shape Contentstack uses for publishing. Filters, sorting and paging go in the query string: /v1/entries?content_type=blog_post&status=published&limit=20.

```bash
# The five REST verbs against one resource, with only the status shown.
# -s hides the progress bar, -o /dev/null throws the body away and
# -w prints a template after the response: here just the status code.
BASE=http://127.0.0.1:8022/v1
KEY="X-API-Key: acme-dev-key"
JSON="Content-Type: application/json"

# Read the collection (safe: changes nothing)
curl -s -o /dev/null -w "%{http_code}\n" "$BASE/entries" -H "$KEY"
# -> 200

# Create: POST to the collection; the server picks the uid
curl -s -i -X POST "$BASE/entries" -H "$KEY" -H "$JSON" \
  -d '{"title": "Navratri lookbook", "content_type": "blog_post"}' \
  | grep -E "^HTTP|^Location"
# -> HTTP/1.1 201 Created
# -> Location: /v1/entries/blt009     <- where the new entry lives

# Partial update: PATCH sends only the fields that change
curl -s -X PATCH "$BASE/entries/blt009" -H "$KEY" -H "$JSON" \
  -d '{"title": "Navratri lookbook 2026"}' | grep -E '"(title|version)"'
# ->   "title": "Navratri lookbook 2026",
# ->   "version": 2,

# An action that is not plain CRUD: a sub-resource you POST to
curl -s -X POST "$BASE/entries/blt009/publish" -H "$KEY" \
  | grep '"status"'
# ->   "status": "published",

# Delete twice: the result on the server is the same, the code is not
curl -s -o /dev/null -w "%{http_code}\n" -X DELETE \
  "$BASE/entries/blt009" -H "$KEY"
# -> 204    <- deleted, no body
curl -s -o /dev/null -w "%{http_code}\n" -X DELETE \
  "$BASE/entries/blt009" -H "$KEY"
# -> 404    <- already gone; still idempotent (the state did not change)
```

### Real-world FDE example

A customer's marketing-ops team wants a small API in front of their CMS so a partner agency can create draft landing pages without CMS logins. A design they can read on one page: GET /v1/pages, POST /v1/pages (always creates a draft), PATCH /v1/pages/{uid}, POST /v1/pages/{uid}/submit-for-review. The agency builds against the design before your code exists, and the customer's security team can see at a glance that nothing in it publishes or deletes.

### Common mistakes

- Verbs in paths: /getEntries, /entries/create. The method is the verb; the path is the noun.

- Using GET for anything that changes data, for example GET /entries/blt001/delete. Link preview bots and prefetching will trigger it.

- Using PUT for a partial update. A PUT with only {"title": ...} means "the entry is now only a title", and a strict server wipes the other fields.

- Nesting deeper than one level (/authors/7/entries/blt001/tags/3). Use a top-level resource plus a filter instead.

- Mixing plural and singular (/entry/1 next to /authors). Pick plural nouns and keep them.

## Concept 3: Status codes and error bodies

### Why it matters for an FDE

The status code is the contract between your API and every client that calls it. Client code, retry logic, monitoring dashboards and alerting all branch on it before anyone reads the body. An API that returns 200 with {"error": "not found"} breaks every one of those tools. In interviews, "which status code would you return here, and why?" is a standard follow-up question in API design rounds.

### How it works

The first digit gives the class: 2xx success, 3xx look elsewhere or use your cached copy, 4xx the client must change something, 5xx the server failed. You need about fifteen codes; learn them with the situation each one means.

| Code | Name | When you return it |
|---|---|---|
| 200 | OK | A successful read or update that returns a body |
| 201 | Created | POST created something; add a Location header with its URL |
| 204 | No Content | Success with nothing to return, usually DELETE |
| 304 | Not Modified | Conditional GET: the client's cached copy is still current (Concept 4) |
| 400 | Bad Request | The request cannot be parsed: broken JSON, an invalid cursor |
| 401 | Unauthorized | We do not know who you are: missing or unknown credentials |
| 403 | Forbidden | We know who you are, and you may not do this |
| 404 | Not Found | No such resource (some APIs also use it to hide resources you may not see) |
| 405 | Method Not Allowed | The URL exists but not with this method; send an Allow header |
| 409 | Conflict | The request clashes with current state, for example a duplicate slug |
| 412 | Precondition Failed | An If-Match condition failed: someone changed it first |
| 415 | Unsupported Media Type | The body is not in a format the server accepts |
| 422 | Unprocessable Content | Valid JSON, but the values break the rules (FastAPI's default for validation) |
| 429 | Too Many Requests | Rate limited; usually with a Retry-After header |
| 500 | Internal Server Error | A bug on the server; the client did nothing wrong |
| 502 / 503 / 504 | Bad Gateway / Unavailable / Gateway Timeout | An upstream or the server itself is down or overloaded; often temporary |

Two pairs cause most confusion. 401 vs 403: 401 is about identity (no key, unknown key, expired token), 403 is about permission (a valid read-only key trying to delete). 400 vs 422: 400 means "I cannot even read this", 422 means "I read it, and the values are wrong". Some APIs use 400 for both; what matters is that you choose once and document it.

The body of an error should have one shape everywhere, so clients can write one error handler. The standard shape is problem details (RFC 9457): a JSON object with type, title, status and detail, sent with Content-Type: application/problem+json, plus any extra fields you need, such as a list of field errors and a request id. The mock API uses it; run the example below and read the bodies it prints.

In [ ]:
# Decide what to do from the status code, the way a careful client does.
import time

import requests  # Day 6; like fetch() but with resp.status_code built in

BASE = "http://127.0.0.1:8022"
KEY = {"X-API-Key": "acme-dev-key"}


def what_to_do(status: int) -> str:
    """Map a status code to the client's next move."""
    if 200 <= status < 300:
        return "success: use the body"
    if status in (408, 429) or status >= 500:
        return "temporary: wait and retry"  # the server may recover
    return "permanent: fix the request, do not retry"  # other 4xx


for path, headers in [
    ("/health", {}),                    # no key needed
    ("/v1/entries", {}),                # forgot the key
    ("/v1/entries/blt999", KEY),        # uid that does not exist
    ("/v1/entries?limit=500", KEY),     # limit outside 1..50
]:
    r = requests.get(BASE + path, headers=headers, timeout=5)
    print(r.status_code, what_to_do(r.status_code))
# -> 200 success: use the body
# -> 401 permanent: fix the request, do not retry
# -> 404 permanent: fix the request, do not retry
# -> 422 permanent: fix the request, do not retry

# Error bodies share one shape, so one code path can log them all.
err = requests.get(BASE + "/v1/entries", timeout=5)
print(err.headers["content-type"])  # -> application/problem+json
print(err.json()["detail"])  # -> Missing or unknown X-API-Key


def get_with_retry(url: str, tries: int = 4) -> requests.Response:
    """GET, retrying 429/503 and honouring the Retry-After header."""
    for attempt in range(1, tries + 1):
        r = requests.get(url, headers=KEY, timeout=5)
        if r.status_code not in (429, 503) or attempt == tries:
            return r
        # Retry-After is in seconds here; fall back to 1 if missing.
        wait = int(r.headers.get("Retry-After", "1"))
        print(f"attempt {attempt}: {r.status_code}, waiting {wait}s")
        time.sleep(wait)
    raise AssertionError("unreachable")


r = get_with_retry(BASE + "/v1/reports/views")
print(r.status_code, r.json())
# -> attempt 1: 503, waiting 1s
# -> attempt 2: 503, waiting 1s
# -> 200 {'views_last_7_days': 18240}

The rule inside what_to_do() is worth memorising: retry 408, 429 and 5xx with a delay; never retry other 4xx. Retrying a 422 sends the same invalid data again and gets the same answer. This is the rule you built into the Day 11 retry decorator, now with a reason for each code.

### Real-world FDE example

An insurance customer's mobile team integrates with your claims API. Their app shows "Something went wrong" for every error because the API returned 500 for validation failures too. You change validation errors to 422 with a field list ({"field": "policy_number", "message": "must be 10 digits"}), and the app can now highlight the wrong field. Support tickets for that screen drop, and the 500 alert on your dashboard means real bugs again.

### Common mistakes

- Returning 200 with an error inside the body. Every HTTP tool will count it as a success.

- Returning 500 for bad input. A 5xx says "our fault"; it pages your on-call engineer and invites retries.

- Mixing up 401 and 403, which sends the customer to fix their key when the real problem is a missing permission.

- A different error shape per endpoint, so every client needs a special case for each one.

- Leaking stack traces or SQL in detail. Log them on the server with the request id; send the client a safe message.

## Concept 4: Headers: content types, auth, caching and safe updates

### Why it matters for an FDE

Headers carry everything that is about the message rather than in it: what format the body is in, who is calling, which version the client already has, how long to cache. Most "works on my machine" integration bugs are header bugs: a missing Content-Type, an Authorization header stripped by a proxy, a cache serving yesterday's content.

### How it works

Header names are case-insensitive (ETag, etag and ETAG are the same header; HTTP/2 sends them all lowercase), and requests gives you a case-insensitive dictionary in r.headers. The ones to know:

| Header | Direction | What it says |
|---|---|---|
| Content-Type | Both | The format of this message's body, such as application/json |
| Accept | Request | The formats the client can read |
| Authorization / X-API-Key | Request | Who is calling: Authorization: Bearer <token>, or an API key header |
| Location | Response | The URL of the resource that a 201 just created |
| ETag | Response | A label for the current version of a resource |
| If-None-Match | Request | Send the body only if the ETag changed (otherwise 304) |
| If-Match | Request | Apply this update only if the ETag is still this one (otherwise 412) |
| Cache-Control | Response | Whether and how long the response may be cached |
| Retry-After | Response | With 429 or 503: how many seconds to wait |
| X-Request-Id | Response | A unique id for this request, to find it in the server logs |

Two of these solve problems you will meet in every content integration. Conditional GET saves work: the client keeps the ETag from the last read and sends it back as If-None-Match; if nothing changed, the server answers 304 with no body. Optimistic concurrency prevents lost updates: two editors read version v1, both edit, and without a check the second save silently overwrites the first. With If-Match: "v1", the second save gets 412 and the client can reload and merge. This is the same idea as the version field in the Day 4 JSON store, moved into HTTP.

```bash
# Conditional requests: ETag for caching, If-Match for safe updates.
BASE=http://127.0.0.1:8022/v1
KEY="X-API-Key: acme-dev-key"

# 1. First read: the server labels this version with an ETag.
curl -s -i "$BASE/entries/blt004" -H "$KEY" | grep -E "^HTTP|^ETag"
# -> HTTP/1.1 200 OK
# -> ETag: "v1"

# 2. Ask again "only if it changed". Same version -> 304, empty body.
#    Single quotes keep the double quotes, which are part of the ETag.
curl -s -i "$BASE/entries/blt004" -H "$KEY" -H 'If-None-Match: "v1"' \
  | head -1
# -> HTTP/1.1 304 Not Modified

# 3. Someone else edits the entry, so its version becomes v2.
curl -s -o /dev/null -X PATCH "$BASE/entries/blt004" -H "$KEY" \
  -H "Content-Type: application/json" -d '{"title": "Returns, made easy"}'

# 4. Our update still says "only if it is still v1". It is refused
#    instead of silently overwriting their change (a "lost update").
curl -s -X PATCH "$BASE/entries/blt004" -H "$KEY" -H 'If-Match: "v1"' \
  -H "Content-Type: application/json" -d '{"title": "My old edit"}' \
  | grep -E '"(status|detail)"'
# ->   "status": 412,
# ->   "detail": "Entry changed: current version is \"v2\"",
```

### Real-world FDE example

A media customer runs a recommendation job that re-reads 40,000 CMS entries every hour, and their CMS plan has a monthly API-call quota. Storing each entry's ETag and sending If-None-Match turns about 39,000 of those calls into 304s with empty bodies, so the job finishes in a tenth of the time. On the same project, two regional teams editing the same product page kept overwriting each other; sending If-Match on every update turned the silent data loss into a clear "someone changed this, reload" message.

### Common mistakes

- Sending JSON without Content-Type: application/json. curl's -d alone sends a form content type, and many servers then reject or misread the body.

- Logging full request headers, which writes API keys and tokens into log files. Mask Authorization and key headers in logs.

- Treating an ETag as a number to compare. It is an opaque label: compare it for equality, quotes included.

- Ignoring Retry-After and retrying immediately, which keeps the rate limit triggered.

## Concept 5: Idempotency keys and pagination

### Why it matters for an FDE

Networks fail in the worst possible way: the request reaches the server, the server creates the entry, and the response is lost. The client sees a timeout and cannot know whether the create happened. Syncs and imports, the core of FDE work, also move far more data than fits in one response. Getting retries and pagination right is the difference between a sync that is correct every night and one that is correct most nights.

### How it works

Idempotency keys. For a POST that must not happen twice (create an entry, charge a card, send an email), the client generates a unique key once per logical operation and sends it in an Idempotency-Key header on every attempt. The server stores the key with the response. When the same key arrives again with the same body, the server returns the stored response instead of creating a second entry. Stripe made this pattern standard, and it is now an IETF draft. Note where the key is created: before the retry loop, not inside it.

In [ ]:
# A retried POST without and with an Idempotency-Key.
import uuid

import requests

URL = "http://127.0.0.1:8022/v1/entries"
KEY = {"X-API-Key": "acme-dev-key"}
entry = {"title": "Winter sale banner", "content_type": "product_page"}


def count_titled(title: str) -> int:
    """How many entries on the server have this title?"""
    r = requests.get(URL, headers=KEY, params={"limit": 50}, timeout=5)
    return sum(e["title"] == title for e in r.json()["items"])


# Imagine the first response was lost (timeout), so the client retries.
for _ in range(2):
    requests.post(URL, headers=KEY, json=entry, timeout=5)
print(count_titled("Winter sale banner"))  # -> 2   <- a duplicate entry

# Same story, but the client sends ONE key for both attempts.
op_key = str(uuid.uuid4())  # made once per logical operation, not per try
entry2 = {"title": "Spring sale banner", "content_type": "product_page"}
for attempt in range(2):
    # json= sets Content-Type: application/json and encodes the dict
    r = requests.post(URL, json=entry2, timeout=5,
                      headers={**KEY, "Idempotency-Key": op_key})
    print(attempt, r.status_code, r.json()["uid"],
          r.headers.get("Idempotent-Replayed"))
# -> 0 201 blt011 None
# -> 1 201 blt011 true    <- same answer replayed, nothing new created
print(count_titled("Spring sale banner"))  # -> 1

Pagination. A list endpoint returns one page at a time. Offset pagination (?limit=20&offset=40) is the LIMIT/OFFSET you wrote on Day 15. It is simple and allows "jump to page 7", but when rows are added or deleted between calls, items shift and you get duplicates or gaps. It also gets slower on deep pages, because the database still walks past every skipped row. Cursor pagination (also called keyset pagination) returns an opaque next_cursor that encodes "after this row"; the next request continues from there. It stays correct while data changes and stays fast, which is why it suits syncs and exports.

In [ ]:
# Why offset pagination breaks while data changes, and how cursors fix it.

# The "server" holds entries sorted newest first, like a CMS list view.
entries = ["e8", "e7", "e6", "e5", "e4", "e3", "e2", "e1"]


def page_by_offset(offset: int, limit: int) -> list[str]:
    """LIMIT/OFFSET from Day 15: skip `offset` rows, take `limit`."""
    return entries[offset:offset + limit]


first = page_by_offset(0, 3)
print(first)  # -> ['e8', 'e7', 'e6']

# Between our two calls an editor publishes a new entry at the top.
entries.insert(0, "e9")

second = page_by_offset(3, 3)
print(second)  # -> ['e6', 'e5', 'e4']   <- e6 appears twice!
# GOTCHA: an item deleted between calls makes one get skipped instead.


def page_by_cursor(after: str | None, limit: int) -> list[str]:
    """Keyset pagination: continue after the last item we saw."""
    start = 0 if after is None else entries.index(after) + 1
    return entries[start:start + limit]


# The cursor is "the last item of the previous page", not a row count,
# so a new entry at the top cannot shift what comes next.
print(page_by_cursor("e6", 3))  # -> ['e5', 'e4', 'e3']
# REMEMBER: offset says "skip N rows"; a cursor says "after this row".

A client that reads every page follows the cursor until the server says there is no next page. The server may also put the next URL in a Link header (rel="next"), which is what GitHub's API does.

In [ ]:
# Fetch every page from the mock API by following next_cursor.
import requests

BASE = "http://127.0.0.1:8022"
KEY = {"X-API-Key": "acme-dev-key"}


def fetch_all(content_type: str, limit: int = 2) -> list[dict]:
    """Collect all entries of one content type, page by page."""
    items: list[dict] = []
    params = {"content_type": content_type, "limit": limit}
    while True:
        # params= builds and encodes the query string for us,
        # like new URLSearchParams(obj) in JS.
        r = requests.get(BASE + "/v1/entries", headers=KEY,
                         params=params, timeout=5)
        r.raise_for_status()  # turn any 4xx/5xx into an exception
        page = r.json()
        items.extend(page["items"])
        print(f"got {len(page['items'])}, next={page['next_cursor']}")
        if page["next_cursor"] is None:  # no more pages
            return items
        params["cursor"] = page["next_cursor"]  # FDE TIP: never decode it


blogs = fetch_all("blog_post")
# -> got 2, next=Ymx0MDAy
# -> got 2, next=None      <- the server knew there was nothing left
print([e["uid"] for e in blogs])
# -> ['blt001', 'blt002', 'blt005', 'blt008']

Two more conventions belong to list endpoints. Always set a default and a maximum page size (the mock uses 3 and 50) so that nobody can ask for a million rows. And put the version in the path (/v1/): adding a field is not a breaking change, but renaming or removing one, or changing a type or a status code, is, and it goes in /v2/ while /v1/ keeps working.

### Real-world FDE example

A customer's nightly export from their product catalogue to a search index pages through 120,000 products with offset pagination while merchandisers keep editing. Every night a few hundred products are missing from search, never the same ones. Switching to a cursor ordered by a stable key (the uid, or updated_at plus uid) removes the gaps. In the same engagement, the order-confirmation webhook retried on timeouts and created duplicate CRM records until each delivery carried an idempotency key that the receiver stored in a unique column: the Day 19 upsert, now in HTTP form.

### Common mistakes

- Generating a new idempotency key on each retry, which makes every attempt look like a new operation.

- Retrying a non-idempotent POST without a key and assuming it is fine because it usually works.

- Using offset pagination for a sync or export over data that changes during the run.

- Building or decoding cursors on the client. A cursor is opaque: pass it back exactly as received.

- No maximum page size, so one careless client can load the whole table into memory.

## Check your understanding (predict first, then run)

Restart the mock (Ctrl+C, then python3 mock_api.py) so the data is fresh. Write the status code you expect for each command, and why, in the notebook before running it. Then run it and explain any surprise in one sentence.

```bash
# Six predictions. Each line prints only the status code.
BASE=http://127.0.0.1:8022/v1
KEY="X-API-Key: acme-dev-key"
W="%{http_code}\n"

# Q1: a read-only key tries to delete an entry
curl -s -o /dev/null -w "$W" -X DELETE "$BASE/entries/blt003" \
  -H "X-API-Key: acme-readonly-key"

# Q2: a JSON body sent with -d but no Content-Type header
curl -s -o /dev/null -w "$W" -X POST "$BASE/entries" -H "$KEY" \
  -d '{"title": "Holi colours", "content_type": "blog_post"}'

# Q3: a page size that is not a number
curl -s -o /dev/null -w "$W" "$BASE/entries?limit=ten" -H "$KEY"

# Q4: PUT on an item (also print the Allow header: use -i and grep)
curl -s -i -X PUT "$BASE/entries/blt001" -H "$KEY" \
  | grep -E "^HTTP|^Allow"

# Q5: publish blt007, then ask for it "only if it is not v1"
curl -s -o /dev/null -X POST "$BASE/entries/blt007/publish" -H "$KEY"
curl -s -o /dev/null -w "$W" "$BASE/entries/blt007" -H "$KEY" \
  -H 'If-None-Match: "v1"'

# Q6: the key header written in lowercase
curl -s -o /dev/null -w "$W" "$BASE/entries/blt001" \
  -H "x-api-key: acme-dev-key"
```

## Hands-on task (about 2 hours): day22/api_design.md and curl_drills.sh

Scenario: Acme Retail (the Contentstack customer from Week 3) wants a small content API in front of the Postgres database you modelled on Day 17 (authors, locales, content_entries, entry_locales, tags, entry_tags). Their partner agency will build against it, so the design has to be agreed before any code is written. From tomorrow you build it with FastAPI, so the design you write today is the spec for Days 23 to 27.

Part A (75 min): fill in day22/api_design.md. The starter file has the eleven sections, a worked example row for one endpoint, and hints in each section. It covers: resources and URLs; an endpoint table; request and response examples; the error format; pagination; filtering and sorting; caching and concurrency (ETag, If-Match); idempotency; authentication; versioning; and open questions for the customer.

Part B (45 min): fill in day22/curl_drills.sh. Each of the 10 drills states a goal against mock_api.py (for example "create an entry and print only its Location header"). Write the command under the drill, run it, and record the status code and a one-line explanation in the ANSWER comment.

Acceptance criteria:

- api_design.md has all 11 sections filled in and grep -c TODO api_design.md prints 0.

- The endpoint table has at least 12 rows covering entries, authors, tags, an entry's tags and an entry's locales; each row has method, path, purpose, success code, main error codes and whether it is idempotent.

- Paths use plural nouns only; the only verb is an action sub-resource such as /publish. No GET changes data.

- Each of 401, 403, 404, 409, 412, 422 and 429 appears at least once with the situation that triggers it.

- The example JSON bodies are valid: each one passes python3 -m json.tool when pasted into a file.

- Pagination uses a cursor, with a stated default and maximum page size and the exact response shape.

- At least three open questions for the customer, each one a decision the design depends on.

- bash curl_drills.sh, run against a freshly started mock, executes all 10 drills top to bottom and each prints what its ANSWER comment says.

- Committed and pushed with a message such as "Day 22: REST design and curl drills" (a WIP push counts).

Hints, not answers:

- Start the design from the customer's tasks, not the tables: "the agency lists published blog posts in Hindi", "an editor attaches the tag sale". Each task becomes one row of the endpoint table.

- For each row ask two questions from today: is it safe, and is it idempotent? The answers tell you the method and whether it needs an Idempotency-Key.

- A duplicate slug or a tag that is already attached is a clash with current state: decide between 409 and a quiet success, and write down why.

- For the drills, curl -s -i ... | grep -E "^HTTP|^Location" shows the status line and one header without the body. -w "%{http_code}" with -o /dev/null shows only the code.

- Drill 10 (the busy report endpoint) needs a small shell loop with sleep; your Day 9 notes on exit codes and loops help.

Stretch (only if the core task and the catch-up item are both pushed): day22/api_client.py has three functions to write with requests: list_all() (follow the cursor), create_once() (POST with an Idempotency-Key and one retry) and get_if_changed() (conditional GET with an ETag).

Catch-up (1h): Day 15's schema.sql and queries.sql, until all 10 queries return the expected results on seed.sql. If Day 4 is still open after yesterday, finish it first.

## DSA practice (30 min)

Both problems are in day22/dsa.py with examples and hints. Give each 15 minutes before reading the hints.

- parse_query: turn a query string such as type=blog_post&tag=sale&tag=diwali&q=steel+bottle into a dict of lists: {'type': ['blog_post'], 'tag': ['sale', 'diwali'], 'q': ['steel bottle']}. Practises splitting strings and a dict of lists (Day 3). Hint: what should happen to flag with no =?

- cursor_page: given sorted uids, a cursor (the last uid of the previous page, or None) and a limit, return the page and the next cursor. It must work even when the cursor's uid has since been deleted. Practises binary search (Day 10) and today's cursor idea. Hint: you want the first uid greater than the cursor.

## Quiz (3 quick questions)

- A client's POST /v1/entries times out and the client retries. What can go wrong, and what are two ways the API design can prevent it?

- Which status code fits each case: no API key; a valid read-only key tries DELETE; a valid key asks for a uid that does not exist?

- An export pages through entries while an editor publishes a new one. Why does cursor pagination stay correct when offset pagination does not?

## What you achieved today

You read HTTP at the byte level, used curl the way support engineers do, and learned the conventions that make an API predictable: nouns and methods, safe and idempotent, status codes with one error shape, conditional headers, idempotency keys and cursors. Your design document is a real FDE artefact: it is the page a customer's team reviews before a build starts.

Tomorrow, Day 23: FastAPI first steps. You build the entries part of today's design as a running API, with routes, Pydantic models, automatic validation errors and interactive /docs.

## When you're done

Paste into the Forward Deployment Engineer project chat as Day 22 review: your api_design.md (or a link to it), the curl_drills.sh output, your six predictions with the real results, your quiz answers, dsa.py, and the link to your latest commit. Say how far the Day 15 catch-up got.

## Recap answers

*Your answers:*
